# WatermarkingForLLM on Google Colab

This notebook builds **CPRF** (Go) and **PRC** (Rust/maturin), installs Python packages, and runs `app.py` using Colab patterns (`%pip`, `pathlib`, `subprocess`).

**VS Code + Google Colab extension:** keep a **`.env`** file in your **local repo root** (see `.env.example`; `.env` is gitignored). Section §1 loads it via `python-dotenv` so **`GITHUB_TOKEN`** (and optional **`HF_TOKEN`**) never live in the notebook. Sync or open the workspace so the runtime can read that file—if the remote VM has no `.env`, fall back to normal **`os.environ`** or (on hosted Colab only) **Colab userdata** secrets.

**Before you start:** **Runtime → Change runtime type → GPU**. Pick a **Python 3.11+** runtime if the menu offers it (required by this project).

In [1]:
import sys

assert sys.version_info >= (3, 11), "Use Python 3.11+ (Runtime → Change runtime type)"

import torch

print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()

Python: 3.12.13
CUDA: True — NVIDIA L4


## 1. Project folder and clone (private repos)

**Secrets (recommended):** copy **`.env.example`** → **`.env`** in the repo root and set **`GITHUB_TOKEN=`** your [GitHub PAT](https://github.com/settings/tokens) (classic **repo** read, or fine-grained **Contents → Read**). Section §1 calls `load_dotenv()` and searches the current working directory and parents for `.env`. Optionally set **`DOTENV_PATH`** below to an absolute path. Tokens are read into **`os.environ`** only (not printed).

**Hosted Colab without `.env`:** you can still use the Colab **Secrets** key **`GITHUB_TOKEN`** as a fallback after env vars.

Edit **`GIT_REPO_OWNER`** and **`GIT_REPO_NAME`** below. The clone URL uses `https://oauth2:<token>@github.com/owner/repo.git` when a token is available.

**Skip cloning:** if you upload/unzip the project yourself, set **`PROJECT_ROOT`** to that folder and **`SKIP_CLONE = True`**.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from urllib.parse import quote

# --- edit for your GitHub repo ---
GIT_REPO_OWNER = "maxraffel"
GIT_REPO_NAME = "attribute-based-watermarking"
SKIP_CLONE = False  # set True if you already put the repo under PROJECT_ROOT

# Optional absolute path to .env (VS Code / synced workspace). If None, search cwd and parents.
DOTENV_PATH: str | None = None

PROJECT_ROOT = Path("/content") / GIT_REPO_NAME


_DOTENV_LOADED = False


def _ensure_dotenv_loaded() -> None:
    global _DOTENV_LOADED
    if _DOTENV_LOADED:
        return
    try:
        from dotenv import load_dotenv
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
            check=True,
        )
        from dotenv import load_dotenv

    candidates: list[Path] = []
    if DOTENV_PATH:
        candidates.append(Path(DOTENV_PATH))
    root = Path.cwd().resolve()
    for _ in range(6):
        candidates.append(root / ".env")
        if root.parent == root:
            break
        root = root.parent
    for p in candidates:
        if p.is_file():
            load_dotenv(p)
            print("Loaded environment from", p)
            _DOTENV_LOADED = True
            return
    print("No .env file found (optional). Set DOTENV_PATH or add .env to the repo root.")
    _DOTENV_LOADED = True


def _github_token() -> str | None:
    _ensure_dotenv_loaded()
    t = os.environ.get("GITHUB_TOKEN", "").strip()
    if t:
        return t
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def _clone_url() -> str:
    token = _github_token()
    base = f"https://github.com/{GIT_REPO_OWNER}/{GIT_REPO_NAME}.git"
    if not token:
        print(
            "No GITHUB_TOKEN in environment: using public HTTPS (private repos will fail). "
            "Add .env or Colab secret GITHUB_TOKEN."
        )
        return base
    return f"https://oauth2:{quote(token, safe='')}@github.com/{GIT_REPO_OWNER}/{GIT_REPO_NAME}.git"


def run_cmd(cmd: list[str] | str, *, cwd: Path | None = None) -> None:
    if isinstance(cmd, str):
        print("$", cmd)
        subprocess.run(cmd, shell=True, check=True, cwd=cwd)
    else:
        print("$", " ".join(cmd))
        subprocess.run(cmd, check=True, cwd=cwd)


if SKIP_CLONE:
    print("SKIP_CLONE=True — not cloning.")
elif not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run_cmd(["git", "clone", "--depth", "1", _clone_url(), str(PROJECT_ROOT)])
else:
    print("Already present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())

No GITHUB_TOKEN: using public HTTPS (private repos will fail). Add Colab secret GITHUB_TOKEN with repo read access.
$ git clone --depth 1 https://github.com/maxraffel/attribute-based-watermarking.git /content/attribute-based-watermarking


CalledProcessError: Command '['git', 'clone', '--depth', '1', 'https://github.com/maxraffel/attribute-based-watermarking.git', '/content/attribute-based-watermarking']' returned non-zero exit status 128.

## 2. Build CPRF (`cprf.so`)

Linux `.so` only — Colab builds it here (your Windows binary is not used). If `go.mod` targets a newer Go than `apt install golang-go` provides, the next cell rewrites the `go` line to **1.23** before `go build`.

In [ ]:
import shutil
import subprocess
from pathlib import Path

cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"

# Colab's apt golang is often older than the repo's `go` directive; relax to 1.23 for the build.
_go_mod = cprf_dir / "go.mod"
if _go_mod.is_file():
    _lines = _go_mod.read_text(encoding="utf-8").splitlines(keepends=True)
    _out, _changed = [], False
    for _line in _lines:
        _s = _line.strip()
        if _s.startswith("go ") and not _s.startswith("go 1.23"):
            _out.append("go 1.23\n")
            _changed = True
        else:
            _out.append(_line)
    if _changed:
        _go_mod.write_text("".join(_out), encoding="utf-8")

if shutil.which("go") is None:
    subprocess.run(
        "apt-get update -qq && apt-get install -qq -y golang-go",
        shell=True,
        check=True,
    )

if not so_path.is_file():
    subprocess.run(
        ["go", "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )

assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)

## 3. Build PRC (Rust + maturin)

Installs Rust in the VM home if `rustc` is missing, then **`maturin build`** (release wheel) and **`pip install`** that wheel. This avoids `maturin develop` issues on hosted Colab; build stdout/stderr are printed when something fails.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

try:
    print("Building Rust package with maturin build...")
    completed_process = subprocess.run(
        ["maturin", "build", "--release", "-m", "prc/Cargo.toml"],
        cwd=PROJECT_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )

    if completed_process.stdout:
        print("Maturin build stdout:\n", completed_process.stdout)
    if completed_process.stderr:
        print("Maturin build stderr:\n", completed_process.stderr)

    if completed_process.returncode != 0:
        raise subprocess.CalledProcessError(
            completed_process.returncode,
            completed_process.args,
            output=completed_process.stdout,
            stderr=completed_process.stderr,
        )

    wheel_dir = PROJECT_ROOT / "prc" / "target" / "wheels"
    wheel_files = sorted(
        wheel_dir.glob("*.whl"), key=lambda p: p.stat().st_mtime, reverse=True
    )
    wheel_file = wheel_files[0] if wheel_files else None

    if wheel_file:
        print(f"Installing {wheel_file.name} with pip...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", str(wheel_file)],
            check=True,
        )
    else:
        raise FileNotFoundError("No wheel file found after maturin build.")

except subprocess.CalledProcessError as e:
    print("Maturin failed with error:", e)
    print("Standard Output:", e.stdout)
    print("Standard Error:", e.stderr)
    raise
except FileNotFoundError as e:
    print(e)
    raise

print("PRC: maturin build and pip install ok")

## 4. Python dependencies (Colab `%pip`)

Uses Colab’s recommended install path. Colab usually ships **PyTorch with CUDA**; extra packages match `pyproject.toml` (without the local-only CUDA index).

In [ ]:
%pip install -q "transformers==5.5.4" "rich>=15" "keybert>=0.9" sentencepiece accelerate python-dotenv

## 5. Hugging Face login (gated Llama)

Accept the **Llama 3.2** license on the model card.

**Non-interactive (VS Code):** add **`HF_TOKEN=`** (or **`HUGGING_FACE_HUB_TOKEN=`**) to your **`.env`**; the next cell logs in from the environment when set.

**Interactive:** if no token is in the environment, the cell falls back to **`notebook_login()`** (paste a token with **read** access).

In [ ]:
import os
from pathlib import Path

from huggingface_hub import login, notebook_login

# Pick up .env from repo root if §1 already chdir'd into PROJECT_ROOT
_env = Path.cwd() / ".env"
if _env.is_file():
    try:
        from dotenv import load_dotenv

        load_dotenv(_env)
    except ImportError:
        pass

_token = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
)
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF: logged in from environment token.")
else:
    notebook_login()

## 6. Pull latest from GitHub

Run this **after §1** whenever you want the newest commit. Reloads **`.env`** from **`PROJECT_ROOT`** if present, then uses **`GITHUB_TOKEN`** from the environment (same source order as §1: `.env` → optional Colab userdata).

If only Python changed, you usually **do not** need to re-run CPRF (§2) or PRC (§3). Re-run those if `cprf/` or `prc/` changed, or if `go build` / `maturin` errors after an update.

In [ ]:
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

# Keep in sync with §1
GIT_REPO_OWNER = "maxraffel"
GIT_REPO_NAME = "attribute-based-watermarking"
PROJECT_ROOT = Path("/content") / GIT_REPO_NAME


def _github_token_for_pull() -> str | None:
    env_path = PROJECT_ROOT / ".env"
    if env_path.is_file():
        try:
            from dotenv import load_dotenv

            load_dotenv(env_path)
            print("Reloaded", env_path)
        except ImportError:
            print("pip install python-dotenv to load .env here")
    t = os.environ.get("GITHUB_TOKEN", "").strip()
    if t:
        return t
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


if not (PROJECT_ROOT / ".git").is_dir():
    raise FileNotFoundError("Run §1 first so PROJECT_ROOT is a git clone.")

os.chdir(PROJECT_ROOT)
tok = _github_token_for_pull()
if tok:
    authed = f"https://oauth2:{quote(tok, safe='')}@github.com/{GIT_REPO_OWNER}/{GIT_REPO_NAME}.git"
    subprocess.run(["git", "remote", "set-url", "origin", authed], check=True)

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True, check=True).stdout.strip())

## 7. Run the demo

Loads **Llama-3.2-1B-Instruct** and **DeBERTa-v3-large** NLI — first run downloads both; VRAM use can be high on a T4.

In [ ]:
import os
import runpy

os.chdir(PROJECT_ROOT)
runpy.run_path(str(PROJECT_ROOT / "app.py"), run_name="__main__")

## 8. Run the policy benchmark

Run **`benchmark_policy_detection.py`** from the repo with parameters you set in the next cell (runs, PRC length, modulus, optional `--reuse-key`, optional extra `--prompt-case id:prompt` lines).

Assume **§1** set `PROJECT_ROOT` and `os.chdir(PROJECT_ROOT)`. Running **§7** first warms HF caches; the benchmark loads the same models again.

In [ ]:
import os
import subprocess
import sys

# --- edit these, then run the cell ---
BENCHMARK_RUNS = 3
BENCHMARK_CODE_LENGTH = 300
BENCHMARK_MODULUS = 1024
BENCHMARK_REUSE_KEY = False  # True → add --reuse-key (one CPRF key per prompt id across runs)
# Optional extra cases: each string is "id:prompt" (only the first ':' splits id from prompt).
# Leave empty to use the script's built-in default prompt cases.
BENCHMARK_EXTRA_PROMPTS: list[str] = []

os.chdir(PROJECT_ROOT)
cmd = [
    sys.executable,
    "benchmark_policy_detection.py",
    "--runs",
    str(BENCHMARK_RUNS),
    "--code-length",
    str(BENCHMARK_CODE_LENGTH),
    "--modulus",
    str(BENCHMARK_MODULUS),
]
if BENCHMARK_REUSE_KEY:
    cmd.append("--reuse-key")
for spec in BENCHMARK_EXTRA_PROMPTS:
    cmd.extend(["--prompt-case", spec])

print("$", " ".join(cmd))
subprocess.run(cmd, check=True)